# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (report_date, client_hash_id, content_hash_id) combination in fact_content_daily_performance, for March 2026 (month=2026-03), used as a mid-panel development month. This table gives day-level granularity per content item per client, which is finer than my original ML-03 framing (one row = one page); I'm aggregating up to page-level for scoring, using this table as the daily source of truth.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: position_tier, content_type, word_count, avg_position
Label/proxy: ctr_gap_ratio = page's own CTR ÷ its position_tier's average CTR (built from observed clicks/impressions only)
Context (not modeled): content_id, client_hash_id, month
Excluded: health_score, priority_score, action_type — golden rule: these are product decision fields, not observed outcomes, and using them as features or labels would let the model cheat by learning a human's past decision instead of a real pattern.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 — grain check
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{table_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(grain_check)
# Result: empty — grain confirmed, no duplicate (date, client, content) combos.

# Query 2 — row count and date span
counts = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{table_path}')
""").df()
print(counts)
# Result: 9,841,378 total rows, spanning 2026-03-01 to 2026-03-31 (full month).

# Query 3 — availability filter
available = con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE
""").df()
print(available)
# Result: 3,611,061 rows survive (36.7% of total) — availability is far from universal.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31
   available_rows
0         3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice's grain confirms cleanly (no duplicate date/client/content combos), but availability is the real constraint: only 3,611,061 of 9,841,378 rows (36.7%) have gsc_data_available = TRUE for March 2026. Any feature or label built from GSC fields (impressions, clicks, position) is only usable for that 36.7% — the rest is either a client without GSC connected (client_has_gsc = False) or a content item with no reportable search activity that day. GA4 coverage is even sparser in the sample rows (ga4_data_available is often <NA> even when GSC is present), meaning any feature mixing GSC and GA4 data would shrink the usable set further. This is a real, measured limitation, not a guess: nearly two-thirds of this table cannot support a GSC-based CTR feature for a given day, so any model trained on this slice implicitly excludes low-visibility content-days, not just "bad" ones.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.



✅ Every section above is filled — markdown thinking AND the code that backs it. Sections 1–4 each have a plain-words answer and a real query or code cell behind it.

✅ The notebook runs top to bottom with no errors (Runtime → Run all). You need to actually verify this yourself — run it fresh end-to-end in Colab before committing, since we built this incrementally across many cells and haven't confirmed a clean full run.

✅ No client names, URLs, or private queries anywhere. All identifiers used (client_hash_id, content_hash_id) are the pseudonymized hash columns — safe per your context brief's own note that hashed IDs are not the "client names/URLs" the rule prohibits.

✅ My claims use careful words: observed, measured, directional, decision-support. The Section 4 write-up says "measured" and "real, measured limitation, not a guess" rather than overclaiming causation.

⬜ Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done. Still on you — commit w03_data_contract.ipynb and paste your repo URL on the assignment card.